In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split, cross_val_score

In [ ]:
data_train = pd.read_csv('/kaggle/input/titanic/train.csv')
data_test = pd.read_csv('/kaggle/input/titanic/test.csv')

In [ ]:
data_train.info()

In [ ]:
data_train.head(10)

In [ ]:
#Очистка пропусков и замена на медианные значения
def prepocess(df):
     df['Age'] = df['Age'].fillna(df['Age'].median())
     df['Fare'] = df['Fare'].fillna(df['Fare'].median())
     df["Sex"] = np.where(df["Sex"] == "male", 1, 0)
     
     emb_map = {'S': 1, 'C': 2, 'Q': 3} 
     df['Embarked'] = df['Embarked'].map(emb_map)
     df['Embarked'] = df['Embarked'].fillna(df['Embarked'].median())


prepocess(data_train)
prepocess(data_test)

In [ ]:
def func_alone(df):
    df["Alone"] = np.where(df["Parch"] + df['SibSp'] >= 1, 0, 1)

func_alone(data_train)
func_alone(data_test)

In [ ]:
data_train = data_train.drop(['Ticket', 'Cabin', 'Name'], axis=1)
data_test = data_test.drop(['Ticket', 'Cabin', 'Name'], axis=1)

In [ ]:
data_train.head(10)

In [ ]:
data_test

In [ ]:
data_train.isnull().sum()

In [ ]:
data_test.isnull().sum()

In [ ]:
data_train.describe()

In [ ]:
#EDA
data_train.hist(bins=15,figsize=(12, 12))
plt.show()

In [ ]:
data_train['Sex'].value_counts().plot.pie(autopct='%0.2f')

In [ ]:
numerical_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Alone']
correlation_matrix = data_train[numerical_cols].corr()
sns.heatmap(correlation_matrix, cmap = 'RdYlGn', annot = True)
plt.show()

In [ ]:
X = data_train.drop(['Survived', 'PassengerId'], axis=1)
y = data_train['Survived']
X_test = data_test.drop(['PassengerId'], axis=1)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state=42)
print(X_train.shape, X_val.shape)

In [ ]:
clf_gb = GradientBoostingClassifier()

In [ ]:
params = {'learning_rate': [0.001, 0.01, 0.1, 1], 'n_estimators': [1, 10, 100, 300, 500], 'max_depth': [1,2,3,4,5,6,7,8]}

In [ ]:
grid_gb = GridSearchCV(estimator=clf_gb, param_grid=params, cv=5)

In [ ]:
grid_gb.fit(X_train, y_train)

In [ ]:
best_gb = grid_gb.best_estimator_
best_gb

In [ ]:
clf_gb = GradientBoostingClassifier(learning_rate=0.01, max_depth=3, n_estimators=300)

In [ ]:
clf_gb.fit(X_train, y_train)

In [ ]:
y_val_xgb = clf_gb.predict(X_val)

In [ ]:
y_val_acc_xgb = accuracy_score(y_val, y_val_xgb)
y_val_acc_xgb

In [ ]:
cv_scores_xgb = cross_val_score(clf_gb, X, y, cv=5, scoring='accuracy')
print(f"Cross-Validation Accuracy: {np.mean(cv_scores_xgb):}")

In [ ]:
y_test_pred = clf_gb.predict(X_test)

In [ ]:
test_pass_id = data_test['PassengerId']

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test_pass_id,
    "Survived": y_test_pred.astype(int)
})
submission.to_csv("gender_submission.csv", index=False)
print("Submission file created successfully!")